# extract feature from ANN models

In [1]:
from torchvision import models
import torch.nn as nn
import numpy as np
import torch
from torch.autograd import Variable as V
from torchvision import transforms as trn
import os
from PIL import Image

In [2]:
# Set random seed for reproducible results
seed = 20200220
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
torch.use_deterministic_algorithms(True)

## alexnet

In [3]:
# select layers of interest and import the pretrained Alexnet model

# lists of alexnet conv and fc layers
conv_layers = ['conv1', 'ReLU1', 'maxpool1', 'conv2', 'ReLU2', 'maxpool2',
    'conv3', 'ReLU3', 'conv4', 'ReLU4', 'conv5', 'ReLU5', 'maxpool5']
fully_connected_layers = ['Dropout6', 'fc6', 'ReLU6', 'Dropout7', 'fc7',
    'ReLU7', 'fc8']

class AlexNet(nn.Module):
    def __init__(self):
        """Select the desired layers and create the model."""
        super(AlexNet, self).__init__()
        self.select_cov = ['maxpool1', 'maxpool2', 'ReLU3', 'ReLU4', 'maxpool5']
        self.select_fully_connected = ['ReLU6' , 'ReLU7', 'fc8']
        self.feat_list = self.select_cov + self.select_fully_connected
        # self.alex_feats = models.alexnet(weights=models.AlexNet_Weights.DEFAULT).features
        self.alex_feats = models.alexnet(pretrained = True).features
        self.alex_classifier = models.alexnet(pretrained = True).classifier
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        self.device = 'cuda' if torch.cuda.is_available() else "cpu"

    def forward(self, x, subp):
        """Extract the feature maps."""
        features = []
        for name, layer in self.alex_feats._modules.items():
            if conv_layers[int(name)] == 'ReLU3':
                out = x.cpu().detach().numpy()
                out_ = out.flatten()
                out_ = out_ - subp
                shp = np.array(out.shape).tolist()
                out_ = out_.reshape(shp)
                output = torch.tensor(out_).float()
                x = output.to(self.device)
            x = layer(x)
            if conv_layers[int(name)] in self.feat_list:
                features.append(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        for name, layer in self.alex_classifier._modules.items():
            x = layer(x)
            if fully_connected_layers[int(name)] in self.feat_list:
                features.append(x)
        return features 

model = AlexNet() # exemplify
if torch.cuda.is_available():
    model.cuda()
model.eval()
# model layer features have no names, just relate to sort in feature list

AlexNet(
  (alex_feats): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (alex_classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(

In [4]:
# image processing
centre_crop = trn.Compose([
    trn.Resize((224,224)),
    trn.ToTensor(),
    trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [10]:
img_set_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\Images')
img_partitions = os.listdir(img_set_dir)

# Create the saving directory if not existing
save_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\derivatives', 'dnn_feature_maps',
        'alexnet')
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for p in img_partitions:
    part_dir = os.path.join(img_set_dir, p) # imageset A-Z
    img_partitions_ = os.listdir(part_dir)
    for pp in img_partitions_:
        part_dir_ = os.path.join(part_dir, pp) # image category aardvark to zucchini
        image_list = []
        for root, dirs, files in os.walk(part_dir_):
            for file in files:
                if file.endswith(".jpg") or file.endswith(".JPEG"):
                    image_list.append(os.path.join(root,file))
        image_list.sort()
        # Extract and save the feature maps
        feats_ = {}
        for i, image in enumerate(image_list):
            img = Image.open(image).convert('RGB')
            input_img = V(centre_crop(img).unsqueeze(0))
            if torch.cuda.is_available():
                input_img=input_img.cuda()
            x = model.forward(input_img)
            feats = {}
            for f, feat in enumerate(x):
                feats[model.feat_list[f]] = feat.data.cpu().numpy()
            feats_[i] = feats
        feat_avg = {}
        for ly in model.feat_list:
            tmp = []
            for ii, image in enumerate(image_list):
                tmp.append(feats_[ii][ly].squeeze())
            tmp_ = np.array(tmp)
            tmp_avg = np.mean(tmp_,0)
            feat_avg[ly] = tmp_avg
        np.save(os.path.join(save_dir, pp), feat_avg)

In [22]:
# how to use
f = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\derivatives\dnn_feature_maps\alexnet', f'acorn.npy')
a = np.load(f,allow_pickle=True)
b = a.tolist()

## resnet50

In [28]:
import torch.utils.model_zoo as model_zoo

In [29]:
# =============================================================================
# Import the model
# =============================================================================
def conv3x3(in_planes, out_planes, stride=1):
	"""3x3 convolution with padding"""
	return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
		padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
	"""1x1 convolution"""
	return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride,
		bias=False)

class BasicBlock(nn.Module):
	expansion = 1  # basic 2 conv layers block

	def __init__(self, inplanes, planes, stride=1, downsample=None):
		super(BasicBlock, self).__init__()
		self.conv1 = conv3x3(inplanes, planes, stride)
		self.bn1 = nn.BatchNorm2d(planes)
		self.relu = nn.ReLU(inplace=True)
		self.conv2 = conv3x3(planes, planes)
		self.bn2 = nn.BatchNorm2d(planes)
		self.downsample = downsample
		self.stride = stride

	def forward(self, x):
		identity = x
		out = self.conv1(x)
		out = self.bn1(out)
		out = self.relu(out)
		out = self.conv2(out)
		out = self.bn2(out)
		if self.downsample is not None:
			identity = self.downsample(x)
		out += identity
		out = self.relu(out)
		return out

class Bottleneck(nn.Module):
	expansion = 4  # 3 conv layers bottleneck block

	def __init__(self, inplanes, planes, stride=1, downsample=None):
		super(Bottleneck, self).__init__()
		self.conv1 = conv1x1(inplanes, planes)
		self.bn1 = nn.BatchNorm2d(planes)
		self.conv2 = conv3x3(planes, planes, stride)
		self.bn2 = nn.BatchNorm2d(planes)
		self.conv3 = conv1x1(planes, planes * self.expansion)
		self.bn3 = nn.BatchNorm2d(planes * self.expansion)
		self.relu = nn.ReLU(inplace=True)
		self.downsample = downsample
		self.stride = stride

	def forward(self, x):
		identity = x
		out = self.conv1(x)
		out = self.bn1(out)
		out = self.relu(out)
		out = self.conv2(out)
		out = self.bn2(out)
		out = self.relu(out)
		out = self.conv3(out)
		out = self.bn3(out)
		if self.downsample is not None:
			identity = self.downsample(x)
		out += identity
		out = self.relu(out)
		return out

class ResNet(nn.Module):

	def __init__(self, block, layers, num_classes=1000, zero_init_residual=False):
		super(ResNet, self).__init__()
		self.inplanes = 64
		self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3,
			bias=False)
		self.bn1 = nn.BatchNorm2d(64)
		self.relu = nn.ReLU(inplace=True)
		self.feat_list = ['block1', 'block2', 'block3', 'block4', 'fc']
		self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
		self.layer1 = self._make_layer(block, 64, layers[0])
		self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
		self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
		self.layer4 = self._make_layer(block, 512, layers[3], stride=2)
		self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
		self.fc = nn.Linear(512 * block.expansion, num_classes)
		for m in self.modules():
			if isinstance(m, nn.Conv2d):
				nn.init.kaiming_normal_(m.weight, mode='fan_out',
					nonlinearity='relu')
			elif isinstance(m, nn.BatchNorm2d):
				nn.init.constant_(m.weight, 1)
				nn.init.constant_(m.bias, 0)
		# Zero-initialize the last BN in each residual branch,
		# so that the residual branch starts with zeros, and each residual
		# block behaves like an identity. This improves the model by 0.2~0.3%
		# according to https://arxiv.org/abs/1706.02677
		if zero_init_residual:
			for m in self.modules():
				if isinstance(m, Bottleneck):
					nn.init.constant_(m.bn3.weight, 0)
				elif isinstance(m, BasicBlock):
					nn.init.constant_(m.bn2.weight, 0)

	def _make_layer(self, block, planes, blocks, stride=1):
		downsample = None
		if stride != 1 or self.inplanes != planes * block.expansion:
			downsample = nn.Sequential(
				conv1x1(self.inplanes, planes * block.expansion, stride),
				nn.BatchNorm2d(planes * block.expansion),
			)
		layers = []
		layers.append(block(self.inplanes, planes, stride, downsample))
		self.inplanes = planes * block.expansion
		for _ in range(1, blocks):
			layers.append(block(self.inplanes, planes))
		return nn.Sequential(*layers)

	def forward(self, x):
		x = self.conv1(x)
		x = self.bn1(x)
		x = self.relu(x)
		x = self.maxpool(x)
		x1= self.layer1(x)
		x2 = self.layer2(x1)
		x3 = self.layer3(x2)
		x4 = self.layer4(x3)
		x = self.avgpool(x4)
		x = x.view(x.size(0), -1)
		x5 = self.fc(x)
		return x1, x2, x3, x4, x5

def resnet50(pretrained=True, **kwargs):
	"""Constructs a ResNet-50 model. """
	model = ResNet(Bottleneck, [3, 4, 6, 3], **kwargs)
	if pretrained == True:
		model_url = 'https://download.pytorch.org/models/resnet50-19c8e357.pth'
		model.load_state_dict(model_zoo.load_url(model_url))
	return model

model = resnet50()
if torch.cuda.is_available():
	model.cuda()
model.eval()

# in this case, the key of feature extraction is that resent.forward() function returns layers' values that we want.

Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to C:\Users\QDZHAO/.cache\torch\hub\checkpoints\resnet50-19c8e357.pth
100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:32<00:00, 3.11MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [30]:
# image processing
centre_crop = trn.Compose([
    trn.Resize((224,224)),
    trn.ToTensor(),
    trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

img_set_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\Images')
img_partitions = os.listdir(img_set_dir)

# Create the saving directory if not existing
save_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\derivatives', 'dnn_feature_maps',
        'resnet50')
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for p in img_partitions:
    part_dir = os.path.join(img_set_dir, p) # imageset A-Z
    img_partitions_ = os.listdir(part_dir)
    for pp in img_partitions_:
        part_dir_ = os.path.join(part_dir, pp) # image category aardvark to zucchini
        image_list = []
        for root, dirs, files in os.walk(part_dir_):
            for file in files:
                if file.endswith(".jpg") or file.endswith(".JPEG"):
                    image_list.append(os.path.join(root,file))
        image_list.sort()
        # Extract and save the feature maps
        feats_ = {}
        for i, image in enumerate(image_list):
            img = Image.open(image).convert('RGB')
            input_img = V(centre_crop(img).unsqueeze(0))
            if torch.cuda.is_available():
                input_img=input_img.cuda()
            x = model.forward(input_img)
            feats = {}
            for f, feat in enumerate(x):
                feats[model.feat_list[f]] = feat.data.cpu().numpy()
            feats_[i] = feats
        feat_avg = {}
        for ly in model.feat_list:
            tmp = []
            for ii, image in enumerate(image_list):
                tmp.append(feats_[ii][ly].squeeze())
            tmp_ = np.array(tmp)
            tmp_avg = np.mean(tmp_,0)
            feat_avg[ly] = tmp_avg
        np.save(os.path.join(save_dir, pp), feat_avg)

## cornet-s

In [18]:
# Import the model
import cornet

def get_model(pretrained=True):
	map_location = None if torch.cuda.is_available() else 'cpu'
	model = getattr(cornet, 'cornet_s')
	model = model(pretrained=pretrained, map_location=map_location)
	if torch.cuda.is_available():
		model = model.cuda()
	else:
		model = model.module # remove DataParallel
	return model



# Define the image preprocessing
centre_crop = trn.Compose([
	trn.Resize((224,224)),
	trn.ToTensor(),
	trn.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


In [27]:
# Load the images and extract the corresponding feature maps
sublayer = 'output'
layers = ['V1', 'V2', 'V4', 'IT', 'decoder']

def _store_feats(layer, inp, output):
	"""An ugly but effective way of accessing intermediate model features
	"""
    # must have three input, including name, input and output
	output = output.detach().numpy()
	_model_feats.append(np.reshape(output, (len(output), -1)))
def get_hook(model,layers,sublayer):
    for layer in layers:
        model_layer = getattr(getattr(model, layer), sublayer)
        model_layer.register_forward_hook(_store_feats)
# hook ref: https://zhuanlan.zhihu.com/p/87853615

img_set_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\Images_')
img_partitions = os.listdir(img_set_dir)

# Create the saving directory if not existing
save_dir = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\derivatives', 'dnn_feature_maps',
        'cornet-s')
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

for p in img_partitions:
    part_dir = os.path.join(img_set_dir, p) # imageset A-Z
    img_partitions_ = os.listdir(part_dir)
    for pp in img_partitions_:
        part_dir_ = os.path.join(part_dir, pp) # image category aardvark to zucchini
        image_list = []
        for root, dirs, files in os.walk(part_dir_):
            for file in files:
                if file.endswith(".jpg") or file.endswith(".JPEG"):
                    image_list.append(os.path.join(root,file))
        image_list.sort()
        # Extract and save the feature maps
        feats_ = {}
        for i, image in enumerate(image_list):
            img = Image.open(image).convert('RGB')
            input_img = V(centre_crop(img).unsqueeze(0))
            if torch.cuda.is_available():
                input_img=input_img.cuda()
            model = get_model()
            model.eval()
            get_hook(model,layers,sublayer)
            # print(model)
            feats = {}
            with torch.no_grad():
                _model_feats = []
                model(input_img) # len(_model_feats)=10 (1+2+4+2+1)
                # Store the feature maps of all time steps
                feats[layers[0]] = _model_feats[0][0] # 200704=64*56*56
                feats[layers[1]] = _model_feats[2][0] # 100352=128*28*28
                feats[layers[2]] = _model_feats[6][0] # 50176=256*14*14
                feats[layers[3]] = _model_feats[8][0] # 25088=512*7*7
                feats[layers[4]] = _model_feats[9][0] # 1000
            feats_[i] = feats
            del model
        feat_avg = {}
        for ly in layers:
            tmp = []
            for ii, image in enumerate(image_list):
                tmp.append(feats_[ii][ly].squeeze())
#             tmp_ = np.array(tmp)
            tmp_avg = np.mean(tmp,0)
            feat_avg[ly] = tmp_avg
        np.save(os.path.join(save_dir, pp), feat_avg)
# code run in extract_feature_cornet-s.py

In [37]:
f = os.path.join(r'F:\visual_project\THINGS_project\THINGS_stimuli\THINGS\derivatives\dnn_feature_maps\cornet-s', f'camel.npy')
a = np.load(f,allow_pickle=True)
b = a.tolist()
print(b['V1'].shape)
print(b['V2'].shape)
print(b['V4'].shape)
print(b['IT'].shape)
print(b['decoder'].shape)

(200704,)
(100352,)
(50176,)
(25088,)
(1000,)
